# Programming Language Salary Analysis Using Web Scraping (Python + BeautifulSoup) & Excel Visualization

In [62]:
# step 1: Install & Import required libraries

!pip install openpyxl

# Web Scraping – Fetching and Parsing Website Content
from bs4 import BeautifulSoup
import requests

# Data Processing and Structuring
import pandas as pd

# Exporting Data to Excel and Creating Charts
from openpyxl import Workbook
from openpyxl.chart import BarChart, Reference 
from openpyxl import load_workbook
from openpyxl.chart.label import DataLabelList


In [63]:
# step 2: Load the Webpage
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DA0321EN-SkillsNetwork/labs/datasets/Programming_Languages.html"

# Download the webpage content as HTML
html = requests.get(url).text
print("Webpage downloaded successfully")



Webpage downloaded successfully


In [64]:
# Step 3 — Create Soup Object
soup= BeautifulSoup(html,"html.parser")
print("Soup object created")

Soup object created


# Extracting Language and Salary Data from the Table, Then Cleaning and Sorting the Dataset

In [65]:
# Step 4 — Extract Language & Salary Data from Table

# 4.1: Extract the Table from the Webpage
table= soup.find("table")

# 4.2: Create Empty Lists to Store Scraped Data
languages =[]
salaries =[]

# 4.3: Loop Through Table Rows and Collect Data
for row in table.find_all("tr")[1:]:   # skip header row
    cols = row.find_all("td")          # Extract All Columns from the Current Row
    languages.append(cols[1].text.strip()) # Save the Programming Language Value
    salaries.append(cols[3].text.strip())  # Save the Salary Value

print("Rows scraped:", len(languages))
print(languages[:5]) # Take the first 5 items
print(salaries[:5])

Rows scraped: 10
['Python', 'Java', 'R', 'Javascript', 'Swift']
['$114,383', '$101,013', '$92,037', '$110,981', '$130,801']


# Key Observations :
    
✔ 10 rows of valid data were successfully scraped

✔ The first 5 languages include Python, Java, R, JavaScript, Swift

✔ Python and Swift appear among the higher-paying languages in the preview

✔ Data quality looks correct (no missing or broken values)

In [66]:
# Step 5: Create DataFrame
df = pd.DataFrame({
    "Language": languages,
    "Average Salary": salaries
})

df


,Language,Average Salary
0,Python,"$114,383"
1,Java,"$101,013"
2,R,"$92,037"
3,Javascript,"$110,981"
4,Swift,"$130,801"
5,C++,"$113,865"
6,C#,"$88,726"
7,PHP,"$84,727"
8,SQL,"$84,793"
9,Go,"$94,082"


# Key Observations :
✔ Data includes 10 languages (Python, Java, R, JavaScript, Swift, etc.)

✔ Salary values are currently stored as text with currency symbols

✔ The DataFrame format is now suitable for:

Converting salary values to numbers

Sorting highest vs lowest salaries

Creating charts in pandas or Excel

Generating insights

In [67]:
# Step 6 :  Clean Currency Symbols

df["Average Salary"]=(
     df["Average Salary"]
     .str.replace("$", "", regex=False)  # Remove the Dollar Symbol ($)
     .str.replace(",", "", regex=False)  # Remove Commas from Numbers
     .str.strip()      # Remove Extra Spaces
)
df.head()
   

,Language,Average Salary
0,Python,114383
1,Java,101013
2,R,92037
3,Javascript,110981
4,Swift,130801


# Key Observations :

The currency symbols and commas were removed from the salary column, converting values such as $114,383 into 114383. 

The data is now clean and properly formatted, making it suitable for numerical operations such as sorting, aggregation, and visualization.

In [68]:
# Step 7 — Convert Salary to Numeric

# 7.1: Convert Salary Column from Text to Numeric Values
df["Average Salary"]= pd.to_numeric(df["Average Salary"],errors="coerce")

# 7.2: Check Data Types and Memory Information
df.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 2 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Language        10 non-null     object
 1   Average Salary  10 non-null     int64 
dtypes: int64(1), object(1)
memory usage: 288.0+ bytes


# Key Observations :

✔ 10 rows in the dataset (RangeIndex: 0–9)

✔ No missing data (10 non-null in each column)

✔ Salary column is now int64 → means true numeric values

✔ Language column is still text (object) → correct

✔ Memory footprint is small → dataset is lightweight

In [69]:
# step 8: Sort by Highest Salary
df=df.sort_values(by="Average Salary", ascending=False)
df


,Language,Average Salary
4,Swift,130801
0,Python,114383
5,C++,113865
3,Javascript,110981
1,Java,101013
9,Go,94082
2,R,92037
6,C#,88726
8,SQL,84793
7,PHP,84727


# Key Observations :

✔ Sorting was applied correctly (highest salary at top)

✔ Top 3 highest-paying languages in this dataset:

Swift, 
Python, 
C++

✔ Lower-paid group in this list:

PHP, 
SQL, 
C#

# Save Data to Excel

In [70]:
# step 9: saves the cleaned & sorted dataframe into an Excel sheet.
file_name = "popular-languages-analysis.xlsx"

with pd.ExcelWriter(file_name, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Language_Salaries", index=False)

print("Step 9 complete — Excel file recreated")

Step 9 complete — Excel file recreated


# Add Bar Chart to Excel

In [71]:
# Step 10: Open the Excel file and insert a bar chart

wb = load_workbook("popular-languages-analysis.xlsx")
sheet = wb["Language_Salaries"]

# Create bar chart
chart = BarChart()
chart.type = "col"
chart.style = 11
chart.title = "Programming Languages by Average Salary"
chart.y_axis.title = "Average Annual Salary (USD)"
chart.x_axis.title = "Programming Language"

# Reset Axis Text Formatting
chart.y_axis.txPr = None
chart.x_axis.txPr = None

# Select data + categories
data = Reference(sheet, min_col=2, min_row=1, max_row=len(df)+1)
categories = Reference(sheet, min_col=1, min_row=2, max_row=len(df)+1)

chart.add_data(data, titles_from_data=True)
chart.set_categories(categories)

# Show values on bars
chart.dLbls = DataLabelList()
chart.dLbls.showVal = True

# Adjust bar spacing
chart.gapWidth = 150

# Add chart to sheet
sheet.add_chart(chart, "E3")

wb.save("popular-languages-analysis.xlsx")
print("Step 10 complete — Color chart added to Excel")


Step 10 complete — Color chart added to Excel
